# Práctica guiada: CNN y Transfer Learning con CIFAR-10

En el workshop clasificamos **piedra 🪨, papel 📄 y tijeras ✂️**. Hoy aplicamos **el mismo pipeline** a un problema más difícil: **CIFAR-10**, 60,000 fotos a color de 32×32 píxeles repartidas en 10 clases (avión, auto, pájaro, gato, venado, perro, rana, caballo, barco y camión).

Entrenaremos y compararemos 3 modelos:

| # | Modelo | Idea clave |
|---|--------|-----------|
| 1 | **LeNet-5** | La CNN clásica del workshop, ahora con 10 clases |
| 2 | **CNN propia** | Una CNN más profunda, diseñada por ti |
| 3 | **Transfer learning (ResNet-18)** | Reutilizar una red ya entrenada con ImageNet |

### El pipeline de siempre

```
Datos →  Preprocesamiento →  DataLoader →  Modelo → Loss → Optimizer → Entrenar → Evaluar
```

### ¿Cómo trabajar este notebook?

- **Bloque A (datos) ya viene resuelto** : solo ejecuta las celdas. La primera vez se descargará CIFAR-10 (~170 MB).
- **Desde el Bloque B**, completa los `# TODO` de cada ejercicio. Los `assert` al final de las celdas te dicen si vas bien: si la celda termina sin error, ¡lo lograste!
- Todo lo que necesitas lo vimos en `36_mlp_cnn` y `37_workshop_cnn`. Si te trabas, revisa esos notebooks.

> 💡 **Stack:** PyTorch · torchvision · timm · NumPy · scikit-learn · matplotlib

## Bloque A — Preparación de los datos (código listo)

### Setup: librerías, semilla y dispositivo

Igual que en el workshop: importamos librerías, fijamos la semilla y elegimos el dispositivo.

Además, definimos `FAST_MODE`:

- `FAST_MODE = True` → usamos una **parte** de CIFAR-10 y pocas épocas, para que la práctica termine en clase (incluso en CPU).
- `FAST_MODE = False` → usamos **todo** el dataset y más épocas. Mejores resultados, pero tarda mucho más (ideal con GPU o en casa).

In [ ]:
from pathlib import Path
import random

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42


def fijar_semilla(seed=SEED):
    """Fija las semillas para que los resultados sean reproducibles."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


fijar_semilla()

if torch.backends.mps.is_available():      # GPU de Apple (Mac M1/M2/M3...)
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():            # GPU NVIDIA
    DEVICE = torch.device("cuda")
else:                                      # CPU
    DEVICE = torch.device("cpu")

FAST_MODE = True

EPOCAS_CNN = 10 if FAST_MODE else 25       # LeNet-5 y CNN propia (desde cero)
EPOCAS_TL = 3 if FAST_MODE else 5          # ResNet-18: solo la capa final
EPOCAS_FT = 3 if FAST_MODE else 5          # ResNet-18: fine-tuning completo (bonus)

print(f"PyTorch {torch.__version__} | Entrenaremos en: {DEVICE}")
print(f"FAST_MODE={FAST_MODE} | épocas CNN={EPOCAS_CNN}, TL={EPOCAS_TL}, FT={EPOCAS_FT}")

### Descargar CIFAR-10

En el workshop leímos imágenes de carpetas con `ImageFolder`. CIFAR-10 es tan conocido que **torchvision trae una clase lista** que lo descarga y lo lee: `datasets.CIFAR10`.

- `train=True` → las 50,000 imágenes oficiales de entrenamiento.
- `train=False` → las 10,000 imágenes oficiales de prueba.
- `download=True` → si no están en `DATA_DIR`, las descarga (solo la primera vez).

Sin `transform`, cada elemento es una tupla `(imagen PIL, etiqueta)`: perfecto para mirar los datos.

> ⚡ El servidor oficial de CIFAR-10 (Universidad de Toronto) suele ser **muy lento** (¡puede tardar horas!). Por eso descargamos el **mismo archivo** desde un espejo en HuggingFace (~10 segundos). torchvision verifica su huella MD5, así que tenemos la garantía de que es idéntico al original.

In [ ]:
# DATA_DIR = Path("/content/data")      # ← descomenta esta línea si trabajas en Google Colab
DATA_DIR = Path("../data")              # carpeta data/ del repositorio

# Espejo rápido del archivo oficial cifar-10-python.tar.gz (mismo MD5)
datasets.CIFAR10.url = "https://huggingface.co/datasets/liangnanying/cifar-10-python/resolve/main/cifar-10-python.tar.gz"

cifar_train_raw = datasets.CIFAR10(root=DATA_DIR, train=True, download=True)
cifar_test_raw = datasets.CIFAR10(root=DATA_DIR, train=False, download=True)

CLASES = cifar_train_raw.classes        # nombres originales (en inglés)
CLASES_ES = ["avión", "auto", "pájaro", "gato", "venado",
             "perro", "rana", "caballo", "barco", "camión"]

imagen, etiqueta = cifar_train_raw[0]
print(f"Train oficial: {len(cifar_train_raw):,} imágenes | Test oficial: {len(cifar_test_raw):,} imágenes")
print(f"Cada imagen: {imagen.size} píxeles, modo {imagen.mode}")
print(f"Primera etiqueta: {etiqueta} → {CLASES[etiqueta]} ({CLASES_ES[etiqueta]})")

print("\nImágenes por clase (train oficial):")
conteo = np.bincount(cifar_train_raw.targets)
for i, n in enumerate(conteo):
    print(f"  {i} {CLASES_ES[i]:>8s} → {n:,}")

### Conoce tus datos

**Regla de oro:** antes de entrenar, *mira* tus datos. Veamos 8 ejemplos de cada clase.

In [ ]:
etiquetas_train = np.array(cifar_train_raw.targets)

fig, axes = plt.subplots(10, 8, figsize=(10, 13))
for clase in range(10):
    indices = np.flatnonzero(etiquetas_train == clase)[:8]
    for col, idx in enumerate(indices):
        axes[clase, col].imshow(cifar_train_raw[idx][0])
        axes[clase, col].axis("off")
    axes[clase, 0].set_title(CLASES_ES[clase], loc="left", fontweight="bold", fontsize=10)
plt.tight_layout()
plt.show()

### ¿Qué "ve" la computadora?

- Cada imagen mide **32 × 32 píxeles** con **3 canales** (RGB) → 32 × 32 × 3 = **3,072 números por imagen**.
- Las imágenes de piedra-papel-tijeras eran de 300 × 200 (¡180,000 números!), pero tenían **fondo verde** y la mano siempre centrada.

**Pregunta breve:** si CIFAR-10 tiene imágenes más pequeñas, ¿por qué es un problema **más difícil**? Piensa en el número de clases, los fondos, las posiciones y lo parecidas que son algunas clases (¿gato vs. perro? ¿auto vs. camión?).

### Preprocesamiento: dos "recetas" de transformación

Igual que en el workshop, cada tipo de modelo necesita su propia receta:

| Modelos | Transformaciones | ¿Por qué? |
|---------|-----------------|-----------|
| LeNet-5 y CNN propia | `ToTensor` → `Normalize(0.5, 0.5)` | Las imágenes ya miden 32×32; solo pasamos a tensor y centramos en [-1, 1] |
| ResNet-18 (transfer learning) | `Resize` → `ToTensor` → `Normalize(ImageNet)` | ResNet se entrenó con fotos grandes (224×224) y con la media/desviación de ImageNet |

> ⏱️ En `FAST_MODE` agrandamos a **128×128** en lugar de 224×224: ResNet sigue funcionando bien y el entrenamiento es ~3 veces más rápido.

In [ ]:
IMG_SIZE_TL = 128 if FAST_MODE else 224

transform_cnn = transforms.Compose([
    transforms.ToTensor(),                       # imagen → tensor con valores [0, 1]
    transforms.Normalize(mean=[0.5, 0.5, 0.5],   # [0, 1] → [-1, 1]
                         std=[0.5, 0.5, 0.5]),
])

transform_tl = transforms.Compose([
    transforms.Resize((IMG_SIZE_TL, IMG_SIZE_TL)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],   # estadísticas de ImageNet
                         std=[0.229, 0.224, 0.225]),
])

print(transform_cnn)
print(transform_tl)

### Dataset y DataLoader: train / validation / test

CIFAR-10 ya viene separado en *train* oficial y *test* oficial. Nosotros sacamos la **validación** del *train* oficial:

| Split | `FAST_MODE=True` | `FAST_MODE=False` | Sale de... |
|-------|-----------------|------------------|-----------|
| **Train** | 10,000 | 45,000 | train oficial |
| **Validation** | 2,000 | 5,000 | train oficial (imágenes distintas a train) |
| **Test** | 2,000 | 10,000 | test oficial |

**¿Cómo elegimos las imágenes?** Barajamos los índices una sola vez (con semilla) y usamos `Subset(dataset, indices)`, que es una "vista" del dataset que solo entrega esos índices. *(En el workshop, `random_split` nos devolvía justamente objetos `Subset`: por eso existía `test_ds.indices`).*

Como los índices se fijan **antes** de crear los DataLoaders, los 3 modelos se entrenan y evalúan con **exactamente las mismas imágenes**, aunque cada uno use una transformación diferente → comparación justa.

In [ ]:
BATCH_SIZE = 64
N_TRAIN, N_VAL, N_TEST = (10_000, 2_000, 2_000) if FAST_MODE else (45_000, 5_000, 10_000)

# Barajamos los índices UNA sola vez: todos los modelos usarán las mismas imágenes
generador = torch.Generator().manual_seed(SEED)
indices_train_oficial = torch.randperm(len(cifar_train_raw), generator=generador).tolist()
indices_test_oficial = torch.randperm(len(cifar_test_raw), generator=generador).tolist()

IDX_TRAIN = indices_train_oficial[:N_TRAIN]
IDX_VAL = indices_train_oficial[N_TRAIN:N_TRAIN + N_VAL]
IDX_TEST = indices_test_oficial[:N_TEST]


def crear_dataloaders(transform, batch_size=BATCH_SIZE):
    """Crea los DataLoaders de train/val/test de CIFAR-10 con un transform dado."""
    train_oficial = datasets.CIFAR10(root=DATA_DIR, train=True, transform=transform)
    test_oficial = datasets.CIFAR10(root=DATA_DIR, train=False, transform=transform)

    train_ds = Subset(train_oficial, IDX_TRAIN)
    val_ds = Subset(train_oficial, IDX_VAL)
    test_ds = Subset(test_oficial, IDX_TEST)

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size)
    test_dl = DataLoader(test_ds, batch_size=batch_size)
    return train_dl, val_dl, test_dl


train_dl_cnn, val_dl_cnn, test_dl_cnn = crear_dataloaders(transform_cnn)
train_dl_tl, val_dl_tl, test_dl_tl = crear_dataloaders(transform_tl)

print(f"Train: {len(train_dl_cnn.dataset):,} | Val: {len(val_dl_cnn.dataset):,} | Test: {len(test_dl_cnn.dataset):,}")
print("Imágenes por clase en train:", np.bincount(etiquetas_train[IDX_TRAIN]))

### Un batch por dentro

Saquemos **un** batch para ver exactamente qué recibirá el modelo. Para dibujarlo deshacemos la normalización (`img * 0.5 + 0.5`) y reordenamos los ejes (`permute`), como en el workshop.

In [ ]:
imagenes, etiquetas = next(iter(train_dl_cnn))

print(f"Batch de imágenes: {imagenes.shape}  ← [batch, canales, alto, ancho]")
print(f"Batch de etiquetas: {etiquetas.shape}")
print(f"Rango de valores: [{imagenes.min():.2f}, {imagenes.max():.2f}]")

imagenes_tl, _ = next(iter(train_dl_tl))
print(f"Batch para ResNet-18: {imagenes_tl.shape}")

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, img, etiqueta in zip(axes.flat, imagenes, etiquetas):
    img = img * 0.5 + 0.5                  # deshacemos la normalización para visualizar
    ax.imshow(img.permute(1, 2, 0))        # [canales, alto, ancho] → [alto, ancho, canales]
    ax.set_title(CLASES_ES[etiqueta], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

**✅ Datos listos.** Desde aquí empieza tu trabajo: completa los `# TODO`.

## Bloque B — CNN entrenadas desde cero

## Ejercicio 1 — Convolución y *pooling* sobre imágenes a color

**Objetivo:** comprobar cómo cambian canales y tamaño espacial dentro de una CNN.

Sobre las 4 primeras imágenes del batch:

1. Crea `nn.Conv2d(3, 16, kernel_size=3, padding=1)` y aplícale ReLU.
2. Crea `nn.MaxPool2d(2)` y aplícalo a los mapas anteriores.
3. Hazlo dentro de `torch.no_grad()` (solo estamos mirando, no entrenando).

**Pregunta:** ¿por qué se pasa de `(4, 3, 32, 32)` a `(4, 16, 32, 32)` y luego a `(4, 16, 16, 16)`?

In [ ]:
imagenes_muestra = imagenes[:4].to(DEVICE)

# TODO: crea una convolución de 3 → 16 canales, kernel 3 y padding 1 (y pásala a DEVICE)
conv_demo = None
# TODO: crea un MaxPool2d de 2×2
pool_demo = None

with torch.no_grad():
    # TODO: aplica conv_demo → ReLU
    mapas = None
    # TODO: aplica pool_demo sobre los mapas
    mapas_pool = None

print("Entrada:     ", imagenes_muestra.shape)
print("Tras Conv2d: ", mapas.shape)
print("Tras MaxPool:", mapas_pool.shape)

assert mapas.shape == (4, 16, 32, 32)
assert mapas_pool.shape == (4, 16, 16, 16)

# Visualizamos lo que "ve" cada filtro (aún aleatorio) en la primera imagen
fig, axes = plt.subplots(1, 8, figsize=(14, 2.2))
axes[0].imshow((imagenes_muestra[0].cpu() * 0.5 + 0.5).permute(1, 2, 0))
axes[0].set_title("original", fontsize=9)
for i, ax in enumerate(axes[1:]):
    ax.imshow(mapas[0, i].cpu(), cmap="viridis")
    ax.set_title(f"filtro {i}", fontsize=9)
for ax in axes:
    ax.axis("off")
plt.show()

**Idea clave:** El formato es `(batch, canales, alto, ancho)`. Cada filtro produce **un canal** de salida. Con kernel 3 y `padding=1` el tamaño espacial se conserva; el *pooling* 2×2 lo reduce a la mitad.

## Ejercicio 2 — LeNet-5 para 10 clases

**Objetivo:** reutilizar la LeNet-5 del workshop en un problema nuevo.

¡Buenas noticias! Las imágenes de CIFAR-10 ya miden **3×32×32**, justo lo que recibía nuestra LeNet-5. Solo cambia la salida: ahora son **10 clases**.

```
3×32×32 → Conv(6 filtros 5×5) → ReLU → Pool → Conv(16 filtros 5×5) → ReLU → Pool → Flatten → 120 → 84 → 10
```

1. Completa la clase `LeNet5` (extractor + clasificador + `forward`).
2. Completa `contar_parametros(modelo)`.

**Comprobación:** la salida para un batch debe ser `(64, 10)` y el modelo debe tener **62,006** parámetros.

In [ ]:
class LeNet5(nn.Module):
    """LeNet-5 (1998) adaptada: entrada RGB 32×32, ReLU y MaxPool."""

    def __init__(self, num_clases=10):
        super().__init__()
        # TODO: Conv(3→6, kernel 5) → ReLU → MaxPool(2) → Conv(6→16, kernel 5) → ReLU → MaxPool(2)
        self.extractor = None
        # TODO: Flatten → Linear(16*5*5 → 120) → ReLU → Linear(120 → 84) → ReLU → Linear(84 → num_clases)
        self.clasificador = None

    def forward(self, x):
        # TODO: pasa x por el extractor y luego por el clasificador
        pass


def contar_parametros(modelo):
    # TODO: suma p.numel() de todos los parámetros con requires_grad=True
    pass


modelo_lenet = LeNet5().to(DEVICE)
salida = modelo_lenet(imagenes.to(DEVICE))

print(modelo_lenet)
print("\nSalida:", salida.shape)
print(f"Parámetros entrenables: {contar_parametros(modelo_lenet):,}")

assert salida.shape == (imagenes.shape[0], 10)
assert contar_parametros(modelo_lenet) == 62_006

**Idea clave:** La salida son **10 logits** (uno por clase), no probabilidades. La clase predicha es la del logit más alto: `logits.argmax(dim=1)`.

## Ejercicio 3 — Funciones de entrenamiento reutilizables

**Objetivo:** escribir el bucle de entrenamiento **una sola vez** y usarlo con los 3 modelos.

Completa:

- `entrenar_una_epoca(...)`: modo `train()`, y para cada batch los 5 pasos: `zero_grad` → forward → loss → `backward` → `step`. Acumula pérdida y aciertos.
- `evaluar(...)`: modo `eval()`, **sin gradientes**, solo mide pérdida y accuracy.
- `fit(...)`: repite ambas durante varias épocas y guarda el historial.

`graficar_historial` ya viene lista.

**Comprobación:** un modelo **sin entrenar** adivina al azar → accuracy ≈ 10% (1 de 10 clases) y loss ≈ 2.30 (= ln 10).

In [ ]:
def entrenar_una_epoca(modelo, dataloader, criterio, optimizador):
    """Una pasada completa por los datos de entrenamiento."""
    # TODO: pon el modelo en modo entrenamiento
    perdida_total, aciertos = 0.0, 0

    for imagenes, etiquetas in dataloader:
        imagenes = imagenes.to(DEVICE)
        etiquetas = etiquetas.to(DEVICE)

        # TODO: 0. limpiar gradientes anteriores
        # TODO: 1. forward: la red predice (logits)
        # TODO: 2. loss: ¿qué tan mal? (perdida)
        # TODO: 3. backward: calcular gradientes
        # TODO: 4. step: ajustar pesos

        # TODO: suma a perdida_total la pérdida del batch × tamaño del batch
        # TODO: suma a aciertos cuántas predicciones (argmax) coinciden con las etiquetas

    n = len(dataloader.dataset)
    return perdida_total / n, aciertos / n


# TODO: añade el decorador que desactiva los gradientes
def evaluar(modelo, dataloader, criterio):
    """Mide loss y accuracy sin modificar el modelo."""
    # TODO: pon el modelo en modo evaluación
    perdida_total, aciertos = 0.0, 0

    for imagenes, etiquetas in dataloader:
        # TODO: mueve los datos a DEVICE, predice y acumula pérdida y aciertos
        pass

    n = len(dataloader.dataset)
    return perdida_total / n, aciertos / n


def fit(modelo, train_dl, val_dl, criterio, optimizador, epocas):
    """Entrena el modelo y guarda el historial de métricas."""
    historial = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoca in range(1, epocas + 1):
        # TODO: entrena una época y evalúa en validación
        train_loss, train_acc = None, None
        val_loss, val_acc = None, None

        # TODO: guarda las 4 métricas en el historial

        print(f"Época {epoca}/{epocas} | "
              f"train loss: {train_loss:.4f}, acc: {train_acc:.2%} | "
              f"val loss: {val_loss:.4f}, acc: {val_acc:.2%}")

    return historial


def graficar_historial(historial, titulo):
    """Curvas de loss y accuracy durante el entrenamiento (ya viene lista)."""
    epocas = range(1, len(historial["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(epocas, historial["train_loss"], "o-", label="train")
    ax1.plot(epocas, historial["val_loss"], "s-", label="validation")
    ax1.set_xlabel("Época"); ax1.set_ylabel("Loss")
    ax1.set_title(f"{titulo} — Loss (↓ mejor)"); ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(epocas, historial["train_acc"], "o-", label="train")
    ax2.plot(epocas, historial["val_acc"], "s-", label="validation")
    ax2.set_xlabel("Época"); ax2.set_ylabel("Accuracy")
    ax2.set_title(f"{titulo} — Accuracy (↑ mejor)"); ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


# Comprobación con un modelo sin entrenar
criterio = nn.CrossEntropyLoss()
loss_azar, acc_azar = evaluar(LeNet5().to(DEVICE), val_dl_cnn, criterio)
print(f"Modelo sin entrenar → loss: {loss_azar:.4f}, accuracy: {acc_azar:.2%}")

assert 2.0 < loss_azar < 2.6
assert 0.03 < acc_azar < 0.20

**Idea clave:** `model.train()` / `model.eval()` cambian el comportamiento de capas como `Dropout`. `torch.no_grad()` evita calcular gradientes cuando solo medimos: más rápido y usa menos memoria.

## Ejercicio 4 — Entrenar LeNet-5

**Objetivo:** obtener nuestro primer modelo de referencia (*baseline*) en CIFAR-10.

1. Llama a `fijar_semilla()` y crea una LeNet-5 nueva.
2. Usa `CrossEntropyLoss` y `Adam` con `lr=1e-3`.
3. Entrena `EPOCAS_CNN` épocas con `train_dl_cnn` / `val_dl_cnn`.
4. Grafica el historial.

**Pregunta breve:** ¿las curvas muestran subajuste, un ajuste razonable o sobreajuste? ¿En qué te fijas para decidirlo?

In [ ]:
fijar_semilla()

# TODO: crea el modelo, la función de pérdida y el optimizador
modelo_lenet = None
criterio = None
optimizador_lenet = None

# TODO: entrena con fit(...) y grafica el historial
historial_lenet = None

**Idea clave:** Si *train* sigue mejorando pero *validation* se estanca o empeora, el modelo está **memorizando** (sobreajuste). Con 62K parámetros y fotos tan variadas, LeNet-5 se queda corta: necesitamos una red con más capacidad.

## Ejercicio 5 — Diseña tu propia CNN (más profunda)

**Objetivo:** construir una CNN con más filtros y más capas que LeNet-5.

Usaremos **3 bloques** `Conv → ReLU → MaxPool`. Con `padding=1` la convolución no reduce el tamaño; solo el *pooling* lo divide entre 2:

```
3×32×32 ─Conv(32)─Pool→ 32×16×16 ─Conv(64)─Pool→ 64×8×8 ─Conv(128)─Pool→ 128×4×4
        → Flatten (2,048) → Linear(256) → ReLU → Dropout(0.3) → Linear(10)
```

Todas las convoluciones: `kernel_size=3, padding=1`.

1. Completa `CNNPropia` (misma estructura `extractor` / `clasificador` que LeNet-5).
2. Ejecuta el recorrido de formas (ya viene escrito) y verifica que llegas a `128×4×4`.

**Comprobación:** la salida debe ser `(64, 10)` y el modelo debe tener **620,362** parámetros.

In [ ]:
class CNNPropia(nn.Module):
    """CNN de 3 bloques Conv → ReLU → MaxPool para imágenes 3×32×32."""

    def __init__(self, num_clases=10):
        super().__init__()
        # TODO: 3 bloques Conv(kernel 3, padding 1) → ReLU → MaxPool(2) con 32, 64 y 128 filtros
        self.extractor = None
        # TODO: Flatten → Linear(128*4*4 → 256) → ReLU → Dropout(0.3) → Linear(256 → num_clases)
        self.clasificador = None

    def forward(self, x):
        # TODO: extractor y luego clasificador
        pass


modelo_cnn = CNNPropia().to(DEVICE)

# Recorrido capa por capa: así verificamos las formas antes de entrenar
x = imagenes[:4].to(DEVICE)
print(f"{'entrada':>10s} → {tuple(x.shape)}")
with torch.no_grad():
    for capa in modelo_cnn.extractor:
        x = capa(x)
        print(f"{capa.__class__.__name__:>10s} → {tuple(x.shape)}")

salida = modelo_cnn(imagenes.to(DEVICE))
print("\nSalida:", salida.shape)
print(f"Parámetros entrenables: {contar_parametros(modelo_cnn):,} "
      f"(LeNet-5: {contar_parametros(modelo_lenet):,})")

assert x.shape == (4, 128, 4, 4)
assert salida.shape == (imagenes.shape[0], 10)
assert contar_parametros(modelo_cnn) == 620_362

**Idea clave:** Cada bloque **duplica los filtros** y **reduce a la mitad** el tamaño: la red cambia resolución espacial por "cantidad de patrones detectados". El `Dropout` ayuda a frenar el sobreajuste.

## Ejercicio 6 — Entrenar tu CNN

**Objetivo:** reutilizar exactamente el mismo pipeline con otra arquitectura.

1. `fijar_semilla()` y crea una `CNNPropia` nueva.
2. `Adam` con `lr=1e-3`, `EPOCAS_CNN` épocas.
3. Grafica el historial.

**Pregunta breve:** compara con LeNet-5. ¿Cuántos puntos de accuracy en validación ganaste? ¿Aparece sobreajuste antes o después?

In [ ]:
fijar_semilla()

# TODO: crea el modelo y su optimizador (el criterio es el mismo de antes)
modelo_cnn = None
optimizador_cnn = None

# TODO: entrena y grafica
historial_cnn = None

**Idea clave:** No cambiamos ni una línea de `fit`: el pipeline **no depende de la arquitectura**. Basta con que el modelo reciba un batch y devuelva logits por clase.

## Bloque C — Transfer learning con ResNet-18

Nuestras CNN aprenden a "ver" **desde cero** con solo 10,000 imágenes. ResNet-18 ya vio **1.2 millones** de imágenes de ImageNet: ya sabe detectar bordes, texturas, formas... ¡y ImageNet incluye aviones, autos, perros, gatos y barcos!

Recuerda los pasos del workshop:

1. Descargar **ResNet-18** pre-entrenada con `timm`, con una capa final nueva de **10 salidas**.
2. **Congelar** todo el cuerpo (`requires_grad = False`).
3. **Descongelar** solo el clasificador (`get_classifier()`) y entrenar únicamente esa capa.

> ⚠️ Usamos `train_dl_tl` / `val_dl_tl` / `test_dl_tl`: imágenes agrandadas y normalizadas con las estadísticas de ImageNet (ya preparadas en el Bloque A).

## Ejercicio 7 — Cargar ResNet-18 y congelar el cuerpo

**Objetivo:** convertir una red de ImageNet en un clasificador de CIFAR-10.

1. Crea `modelo_tl` con `timm.create_model("resnet18", pretrained=True, num_classes=10)`.
2. Congela **todos** sus parámetros.
3. Descongela solo los de `modelo_tl.get_classifier()`.
4. Pásalo a `DEVICE` y haz un forward con un batch de `imagenes_tl`.

**Comprobación:** solo deben quedar **5,130** parámetros entrenables (512 × 10 pesos + 10 sesgos).

In [ ]:
import timm

fijar_semilla()
# TODO: descarga ResNet-18 pre-entrenada con una capa final nueva de 10 clases
modelo_tl = None

# TODO: 1) congela TODOS los pesos

# TODO: 2) descongela solo los parámetros del clasificador (get_classifier)

modelo_tl = modelo_tl.to(DEVICE)

# TODO: haz un forward con imagenes_tl
salida_tl = None
total = sum(p.numel() for p in modelo_tl.parameters())
print("Clasificador:", modelo_tl.get_classifier())
print("Salida:", salida_tl.shape)
print(f"Parámetros totales:      {total:,}")
print(f"Parámetros entrenables:  {contar_parametros(modelo_tl):,} "
      f"({contar_parametros(modelo_tl) / total:.2%} del total)")

assert salida_tl.shape == (imagenes_tl.shape[0], 10)
assert contar_parametros(modelo_tl) == 512 * 10 + 10

**Idea clave:** ResNet-18 tiene ~11 millones de parámetros, pero entrenamos menos del 0.1%. El cuerpo congelado funciona como un **extractor de características** experto.

## Ejercicio 8 — Entrenar solo la capa final

**Objetivo:** adaptar ResNet-18 a CIFAR-10 con muy poco entrenamiento.

1. Crea un `Adam` con `lr=1e-3` que reciba **solo** los parámetros con `requires_grad=True`.
2. Entrena `EPOCAS_TL` épocas con los DataLoaders `_tl`.
3. Grafica el historial.

**Pregunta breve:** con solo `EPOCAS_TL` épocas, ¿cómo se compara con tus CNN entrenadas desde cero? ¿Por qué necesita tan pocas épocas?

> ⏱️ Cada época tarda más que en las CNN pequeñas: la imagen es más grande y ResNet-18 tiene 18 capas.

In [ ]:
# TODO: optimizador Adam solo con los parámetros que tienen requires_grad=True
optimizador_tl = None

# TODO: entrena con los DataLoaders de transfer learning y grafica
historial_tl = None

**Idea clave:** Solo aprendemos a **combinar** características que ResNet ya sabe extraer. Por eso con pocas épocas supera a modelos entrenados desde cero.

## Ejercicio 9 (Bonus) — Fine-tuning completo

**Objetivo:** ajustar **toda** la red, no solo la capa final.

Igual que en el bonus del workshop:

1. Crea un modelo **nuevo** `modelo_ft` (ResNet-18 pre-entrenada, 10 clases) — no reutilices `modelo_tl`, así podremos comparar ambos.
2. Deja **todos** los parámetros entrenables.
3. Usa `Adam` con un learning rate **muy pequeño**, `lr=1e-4`, para no destruir lo que ya sabe.
4. Entrena `EPOCAS_FT` épocas y grafica.

> ⚠️ Es el entrenamiento más costoso del notebook. En CPU puede tardar bastante: si no hay tiempo, sáltalo y deja fuera `modelo_ft` en el Bloque D.

**Pregunta clave:** ¿por qué se usa un learning rate más pequeño que al entrenar solo la cabeza?

In [ ]:
fijar_semilla()
# TODO: crea un modelo nuevo ResNet-18 pre-entrenado de 10 clases y pásalo a DEVICE
modelo_ft = None

# TODO: asegúrate de que TODOS los parámetros sean entrenables

# TODO: optimizador Adam con lr=1e-4, entrena EPOCAS_FT épocas y grafica
optimizador_ft = None
historial_ft = None

**Idea clave:** En el *fine-tuning* los pesos pre-entrenados ya son buenos; pasos grandes los "romperían". Por eso el learning rate es pequeño y bastan pocas épocas.

## Bloque D — Evaluación final y comparación

## Ejercicio 10 — El examen final: test set

**Objetivo:** evaluar todos los modelos con imágenes que **nunca** vieron.

1. Completa `obtener_predicciones(modelo, dataloader)`: devuelve `(y_real, y_pred)` recorriendo el DataLoader sin gradientes.
2. Calcula la accuracy en test de cada modelo con `accuracy_score`.
3. Dibuja la matriz de confusión de cada modelo (usa `display_labels=CLASES_ES`).
4. Muestra el `classification_report` del mejor modelo.

**Pregunta breve:** ¿qué pares de clases se confunden más? ¿Tiene sentido visualmente?

> Si te saltaste el bonus, borra la línea de `modelo_ft` en `modelos_finales`.

In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
)


# TODO: añade el decorador que desactiva los gradientes
def obtener_predicciones(modelo, dataloader):
    """Devuelve (etiquetas reales, etiquetas predichas) de todo un dataloader."""
    # TODO: modo evaluación; recorre el dataloader, predice con argmax y
    #       acumula etiquetas reales y predichas en dos listas (en CPU)
    y_real, y_pred = [], []
    return y_real, y_pred


modelos_finales = [
    ("LeNet-5", modelo_lenet, test_dl_cnn),
    ("CNN propia", modelo_cnn, test_dl_cnn),
    ("ResNet-18 (TL)", modelo_tl, test_dl_tl),
    ("ResNet-18 (FT)", modelo_ft, test_dl_tl),    # ← bórrala si no hiciste el bonus
]

resultados = {}
predicciones = {}
for nombre, modelo, test_dl in modelos_finales:
    # TODO: obtén las predicciones, guárdalas en `predicciones[nombre]`
    #       y la accuracy (accuracy_score) en `resultados[nombre]`
    print(f"{nombre:>15s} → accuracy en test: {resultados[nombre]:.2%}")

In [ ]:
# TODO: una matriz de confusión por modelo (una fila de subplots), con display_labels=CLASES_ES
#       Tip: usa xticks_rotation=45 para que se lean los nombres


# TODO: classification_report del modelo con mayor accuracy (target_names=CLASES_ES)
mejor_nombre = None

**Idea clave:** La accuracy global esconde detalles. La matriz de confusión y el *recall* por clase muestran **dónde** falla cada modelo (típicamente gato ↔ perro y auto ↔ camión).

## Ejercicio 11 — Comparar los modelos

**Objetivo:** contrastar desempeño y tamaño, no solo accuracy.

1. Dibuja un gráfico de barras con la accuracy en test de cada modelo.
2. Imprime una tabla con: parámetros **totales**, parámetros **entrenados** y accuracy en test.

**Preguntas:**

- ¿Más parámetros implica siempre más accuracy?
- ResNet-18 (TL) tiene ~11M parámetros pero entrena solo ~5K. ¿Qué te dice eso?
- ¿Cuándo preferirías una CNN pequeña entrenada desde cero? (piensa en un celular o en imágenes muy distintas a ImageNet, como radiografías)

In [ ]:
# TODO: gráfico de barras con la accuracy en test de cada modelo (usa `resultados`)
#       Tip: ax.bar_label(barras, fmt=lambda v: f"{v:.1%}") pone el valor sobre cada barra


# TODO: imprime por modelo: parámetros totales, parámetros entrenados y accuracy en test
#       Tip: totales → sum(p.numel() for p in modelo.parameters())
#            entrenados → contar_parametros(modelo)

**Idea clave:** La comparación es empírica (puede variar con otra semilla o más épocas). Aun así, la lección se repite: con pocos datos y poco tiempo, **reutilizar conocimiento** (transfer learning) suele ganar.

## Ejercicio 12 — Pruébalo tú: predice una imagen

**Objetivo:** usar el modelo como lo harías en producción: una imagen → una predicción con confianza.

Completa `predecir(imagen_pil, modelo, transform)`:

1. Aplica `transform` y añade la dimensión de batch con `unsqueeze(0)`.
2. Calcula probabilidades con `torch.softmax(..., dim=1)`.
3. Muestra la imagen con la clase predicha y su confianza, y el top-3 de clases.

Las imágenes salen de `cifar_test_raw` (sin transformar) usando índices de `IDX_TEST`.

In [ ]:
# TODO: añade el decorador que desactiva los gradientes
def predecir(imagen_pil, modelo=modelo_tl, transform=transform_tl):
    """Clasifica una imagen PIL y la muestra junto a la predicción."""
    # TODO: modo evaluación
    # TODO: transforma la imagen, añade la dimensión de batch y pásala a DEVICE
    tensor = None
    # TODO: probabilidades con softmax (quédate con la fila [0]) y clase predicha con argmax
    probabilidades = None
    prediccion = None

    plt.figure(figsize=(3, 3))
    plt.imshow(imagen_pil)
    plt.axis("off")
    plt.title(f"Predicción: {CLASES_ES[prediccion]} ({probabilidades[prediccion]:.1%})")
    plt.show()

    top3 = probabilidades.topk(3)
    for prob, clase in zip(top3.values, top3.indices):
        print(f"  {CLASES_ES[clase]:>8s}: {prob:.1%}")


for idx in random.sample(IDX_TEST, 3):
    imagen, etiqueta = cifar_test_raw[idx]
    print(f"Clase real: {CLASES_ES[etiqueta]}")
    predecir(imagen)

**Reto opcional:** llama a `predecir(imagen, modelo=modelo_cnn, transform=transform_cnn)` con la misma imagen. ¿Ambos modelos están igual de seguros?

## Ejercicio 13 — Guardar y recargar los modelos

**Objetivo:** no perder el entrenamiento al cerrar el notebook.

1. Guarda el `state_dict` de `modelo_cnn` y `modelo_tl` en `../models/` (`cnn_propia_cifar10.pt` y `resnet18_cifar10.pt`).
2. Recarga la CNN: crea una `CNNPropia()` nueva, carga los pesos con `map_location=DEVICE` y activa `eval()`.
3. Verifica que la CNN recargada obtenga **la misma accuracy** en test que la original.

In [ ]:
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)

# TODO: guarda los state_dict de modelo_cnn y modelo_tl en MODELS_DIR


for archivo in sorted(MODELS_DIR.glob("*cifar10.pt")):
    tamano_mb = archivo.stat().st_size / 1024**2
    print(f"✅ {archivo.name} ({tamano_mb:.1f} MB)")

# TODO: crea una CNNPropia nueva, carga sus pesos (map_location=DEVICE), pásala a DEVICE y a eval()
cnn_cargada = None

# TODO: calcula su accuracy en test con obtener_predicciones + accuracy_score
acc_cargada = None
print(f"Accuracy de la CNN cargada desde disco: {acc_cargada:.2%} (original: {resultados['CNN propia']:.2%})")

assert acc_cargada == resultados["CNN propia"]

**Idea clave:** `state_dict` guarda solo los **pesos**, no la clase. Para recargarlo necesitas volver a construir **la misma arquitectura** (por eso la clase `CNNPropia` debe estar definida).

## Cierre: ¿qué aprendimos?

| | LeNet-5 | CNN propia | ResNet-18 (TL) | ResNet-18 (FT) |
|---|---|---|---|---|
| **Idea** | CNN clásica y pequeña | Más filtros y más capas | Reutiliza una red experta | Ajusta toda la red experta |
| **Entrada** | 32×32 | 32×32 | 128×128 (224 en modo completo) | igual que TL |
| **Parámetros entrenados** | ~62K | ~620K | ~5K de ~11M | ~11M |
| **Conocimiento previo** | Ninguno | Ninguno | ImageNet | ImageNet |

### Para tu conclusión, responde:

1. ¿Qué modelo obtuvo la mejor accuracy en test? ¿Y en qué clases mejoró más respecto a LeNet-5?
2. ¿Qué modelo tardó más por época en tu computadora? ¿Compensa la mejora?
3. ¿Qué crees que pasaría con `FAST_MODE = False` (4.5 veces más datos y más épocas)?
4. ¿Por qué una comparación seria debería repetirse con varias semillas?

### Próximos pasos (temas que veremos más adelante)

- **Data augmentation** (`RandomCrop`, `RandomHorizontalFlip`...): generar variaciones de las imágenes para reducir el sobreajuste.
- **Batch Normalization**: normalizar dentro de la red para entrenar redes más profundas y estables.
- **Learning rate schedulers**: bajar el learning rate a medida que avanza el entrenamiento.
- **Early stopping**: guardar el modelo de la mejor época de validación.
- Probar otras redes de `timm`: `efficientnet_b0`, `mobilenetv3_small_100`...